In [68]:
import pandas as pd
import numpy as np
import ast

In [69]:
filepath ='usersround2/processed_data'
labeledComments = pd.read_csv(f'{filepath}/labeledDataClean.csv')
participantsAll = pd.read_csv(f'{filepath}/participants.csv')
allComments = pd.read_csv(f'{filepath}/valid_comments.csv')
myPosts = pd.read_json('posts.json', lines=True)
myPosts['_id'] = myPosts['_id'].apply(lambda x: x.get('$oid', None))

In [70]:
validParticipants = set(allComments['user_id']) # recall we dropped those that somehow submitted less than four comments
participantMask = participantsAll['user_id'].isin(validParticipants)
participantsValid = participantsAll[participantMask]

In [71]:
labeledComments['comment'] = labeledComments['comment'].apply(str)
allComments['comment'] = allComments['comment'].apply(str)
mergedComments = allComments.merge(labeledComments, on='comment')

In [72]:
myMap = {'IH': 1, 'Neutral': 0, 'IA': -1}

smallDf = mergedComments[['user_id', 'posted', 'classification_string', 'comment']].copy()
smallDf['demonstratedIH'] = smallDf['classification_string'].map(myMap)
smallDf['commentLength'] = smallDf['comment'].apply(len)
onlyPosted = smallDf[smallDf.posted]

In [73]:
onlyPosted

,user_id,posted,classification_string,comment,demonstratedIH,commentLength
0,60ff181e02cb6fc7912b2203,True,IH,I understand the concerns about strained resou...,1,728
1,60ff181e02cb6fc7912b2203,True,Neutral,1000% agree. We need to look at the government...,0,63
2,5bb58e338f3bd70001e5df57,True,IH,the thing is at the end of the day it is a wom...,1,85
4,6669fe44c5660a00d14ab11d,True,IH,"This is all valid, but, adding more options fo...",1,232
5,60ff181e02cb6fc7912b2203,True,Neutral,These people are looking to improve their and ...,0,170
...,...,...,...,...,...,...
3412,5c70eafaf6c9fe00013eeafb,True,IH,"If we're only using anecdotal evidence, I know...",1,121
3413,5c70eafaf6c9fe00013eeafb,True,Neutral,All choices will have a positive or negative o...,0,153
3414,5c70eafaf6c9fe00013eeafb,True,Neutral,You're repetitive,0,17
3415,5c70eafaf6c9fe00013eeafb,True,IA,Try reading.,-1,12


In [74]:

demonstratedIH = onlyPosted.groupby('user_id').agg({'demonstratedIH':np.mean, 
                                                    'commentLength':np.mean, 
                                                    'comment':np.count_nonzero})

/var/folders/04/dgz4d5cn0v732mbzpk64d_rc0000gn/T/ipykernel_99527/2116774943.py:1: FutureWarning: The provided callable <function mean at 0x10c5f23e0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  demonstratedIH = onlyPosted.groupby('user_id').agg({'demonstratedIH':np.mean,
/var/folders/04/dgz4d5cn0v732mbzpk64d_rc0000gn/T/ipykernel_99527/2116774943.py:1: FutureWarning: The provided callable <function mean at 0x10c5f23e0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  demonstratedIH = onlyPosted.groupby('user_id').agg({'demonstratedIH':np.mean,


In [75]:
myPostsJoin = myPosts[['_id', 'stance']].rename(columns={'_id':'post1'})

In [76]:
participantsWViews = participantsValid.merge(myPostsJoin, right_on='post1', left_on='post1')
participantsWViews.set_index('user_id', inplace=True)

In [77]:
fullDf = participantsWViews.join(demonstratedIH)

In [78]:
fullDf = fullDf.reset_index()

In [80]:
finalDf = fullDf[['social_cue_nudge', 'environment_condtion', 'actually_treated', 'pre_IH', 'post_IH', 'stance', 'demonstratedIH', 
                   'commentLength', 'comment']]

In [81]:
finalDf

,social_cue_nudge,environment_condtion,actually_treated,pre_IH,post_IH,stance,demonstratedIH,commentLength,comment
0,False,NEUTRAL,False,7.375,6.875,Undocumented Immigrant Rights and Immigration ...,0.166667,294.666667,6
1,True,NEUTRAL,True,7.250,7.125,Abortion and Reproductive Rights - FOR,0.500000,159.666667,6
2,False,IA,False,6.500,6.250,Abortion and Reproductive Rights - FOR,0.000000,142.166667,6
3,True,IH,True,9.000,9.000,Abortion and Reproductive Rights - AGAINST,0.714286,60.571429,7
4,False,IA,False,6.375,6.000,Action for Climate Change - AGAINST,0.571429,172.000000,7
...,...,...,...,...,...,...,...,...,...
350,True,IH,True,4.750,4.875,Action for Climate Change - AGAINST,0.125000,355.750000,8
351,False,IA,False,8.000,5.750,Action for Climate Change - AGAINST,-0.571429,164.714286,7
352,False,NEUTRAL,False,8.000,8.000,Action for Climate Change - AGAINST,0.888889,376.333333,9
353,False,IA,False,7.750,8.375,Action for Climate Change - AGAINST,0.375000,196.875000,8


In [82]:
finalDf.to_csv(f'{filepath}/output.csv')

In [52]:
# Reasons people gave
participantsWViews['nextPost'] = participantsWViews['nextPostReason'].apply(lambda x: [i[0] for i in list(ast.literal_eval(x).values())])

In [53]:
allReasons = list(participantsWViews['nextPost'])
reasons = set()
for i in allReasons:
    for j in i:
        reasons.add(j)
# Create one-hot columns
for reason in reasons:
    participantsWViews[reason] = participantsWViews['nextPost'].apply(lambda x: int(reason in x))


In [54]:
mapping = {}
for i in reasons:
    mapping[i] = np.count_nonzero

mapping['pre_IH'] = np.count_nonzero

participantsWViews.groupby(['environment_condtion', 'social_cue_nudge']).agg(mapping)

KeyError: "Column(s) ['pre_IH'] do not exist"

In [17]:
prolificInfo1 = pd.read_csv(f"usersround1/prolificParticipantInfo.csv")
prolificInfo2 = pd.read_csv(f"usersround2/prolificParticipantInfo.csv")
prolificInfo = pd.concat([prolificInfo1, prolificInfo2]).reset_index()
prolificInfo.columns

Index(['index', 'Submission id', 'Participant id', 'Status',
       'Custom study tncs accepted at', 'Started at', 'Completed at',
       'Reviewed at', 'Archived at', 'Time taken', 'Completion code',
       'Total approvals', 'Fluent languages', 'Social-media',
       'U.s. political affiliation', 'Age', 'Sex', 'Ethnicity simplified',
       'Country of birth', 'Country of residence', 'Nationality', 'Language',
       'Student status', 'Employment status'],
      dtype='object')

In [18]:
wDemo = finalDf.merge(prolificInfo[['Participant id', 'Age', 'Sex', 'U.s. political affiliation']], left_on='user_id', right_on='Participant id')

In [19]:
wDemo.to_csv(f'{filepath}/output_w_demo.csv')

In [20]:
import pandas as pd
import numpy as np
comments = pd.read_csv('usersround2/processed_data/output.csv')

In [30]:
comments.groupby('social_cue_nudge').agg({'comment':np.mean})

/var/folders/04/dgz4d5cn0v732mbzpk64d_rc0000gn/T/ipykernel_70982/1646066528.py:1: FutureWarning: The provided callable <function mean at 0x10bd02020> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  comments.groupby('social_cue_nudge').agg({'comment':np.mean})


,comment
social_cue_nudge,
False,7.322222
True,7.194286


,user_id,social_cue_nudge,environment_condtion,actually_treated,pre_IH,post_IH,stance,demonstratedIH,commentLength,comment
0,60ff181e02cb6fc7912b2203,False,NEUTRAL,False,4.625,4.125,Undocumented Immigrant Rights and Immigration ...,0.166667,294.666667,6
1,673e2d50969a90059ceee55d,True,NEUTRAL,True,4.500,4.375,Abortion and Reproductive Rights - FOR,0.500000,159.666667,6
2,67ba35b0a2aff76b7c7beae0,False,IA,False,3.750,3.500,Abortion and Reproductive Rights - FOR,0.000000,142.166667,6
3,5bb58e338f3bd70001e5df57,True,IH,True,6.250,6.250,Abortion and Reproductive Rights - AGAINST,0.714286,60.571429,7
4,67f1cc26973bd7f41512c209,False,IA,False,3.625,3.250,Action for Climate Change - AGAINST,0.571429,172.000000,7
...,...,...,...,...,...,...,...,...,...,...
350,5b57d2acbc04d60001e5e76e,True,IH,True,2.000,2.125,Action for Climate Change - AGAINST,0.125000,355.750000,8
351,63d82af997cbffac143bddb0,False,IA,False,5.250,3.000,Action for Climate Change - AGAINST,-0.571429,164.714286,7
352,65a1ffc87be6e728c2c207c9,False,NEUTRAL,False,5.250,5.250,Action for Climate Change - AGAINST,0.888889,376.333333,9
353,5e7d68d6f772be3f923ea8fd,False,IA,False,5.000,5.625,Action for Climate Change - AGAINST,0.375000,196.875000,8


In [83]:
oldData = pd.read_csv('usersround2/processed_data/output-06-2025.csv')
newData = pd.read_csv('usersround2/processed_data/output-06-2025.csv')

In [87]:
oldData == newData

,user_id,social_cue_nudge,environment_condtion,actually_treated,pre_IH,post_IH,stance,demonstratedIH,commentLength,comment
0,True,True,True,True,True,True,True,True,True,True
1,True,True,True,True,True,True,True,True,True,True
2,True,True,True,True,True,True,True,True,True,True
3,True,True,True,True,True,True,True,True,True,True
4,True,True,True,True,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...
350,True,True,True,True,True,True,True,True,True,True
351,True,True,True,True,True,True,True,True,True,True
352,True,True,True,True,True,True,True,True,True,True
353,True,True,True,True,True,True,True,True,True,True


0      4.625
1      4.500
2      3.750
3      6.250
4      3.625
       ...  
350    2.000
351    5.250
352    5.250
353    5.000
354    4.000
Name: pre_IH, Length: 355, dtype: float64